In [1]:
import sqlite3
conn = sqlite3.connect('english_lesson.db')
cursor = conn.cursor()

In [32]:
 cursor.execute('delete from word_book_words where word_book_id=1 and word_id > 1775')

In [33]:
conn.commit()

In [9]:
import pymysql
import pandas as pd
from pymysql.cursors import DictCursor
# 连接 MySQL 数据库
conn_prod = pymysql.connect(
    host='pxc-hzssyaotpmq5wt-pub.polarx.rds.aliyuncs.com',       # 主机地址
    port=3306,
    user='humian',   # 用户名
    password='XSbVZKGndT&+<M5$;9^S',  # 密码
    database='bfst',     # 数据库名
    cursorclass=DictCursor
)

# conn = pymysql.connect(
#     host='rm-wz910e8b286009p2wzo.mysql.rds.aliyuncs.com',       # 主机地址
#     port=3306,
#     user='root',   # 用户名
#     password='A123456a',  # 密码
#     database='bfst',     # 数据库名
#     ssl={}
# )
cursor_prod = conn_prod.cursor()

# 添加单词本

## 单词本创建

In [34]:
cursor.execute(
    'INSERT INTO word_books (name, description, icon, word_count) VALUES (?, ?, ?, ?)',
    ('计算机AI', '计算机领域和AI领域专业词汇', '🏠', 0)
)

In [35]:
conn.commit()

## 单词本添加单词

In [25]:
import json

# 如果每行是一个独立的 JSON 对象
words_data = []
with open(r"C:\Users\17245\Downloads\english_dict_data\CET6_2.json", 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:  # 跳过空行
            try:
                words_data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"跳过无效行: {e}")

In [26]:
words = []
for data in words_data:
    words.append(data['headWord'])

In [36]:
words_str = '''algorithm architecture assembler bandwidth binary bit boolean buffer bus byte cache chip clock cluster compiler computation concurrency configuration console container core cpu cryptography daemon database debugger decryption dependency deployment descriptor directory distributed driver encoding encryption endpoint execution filesystem firmware framework gateway gpu hardware hash heap hypervisor indexing instruction interrupt ioctl kernel latency library linker loadbalancer localhost logging machinecode mainframe memory metadata middleware migration multithreading namespace network node opcode operatingsystem orchestration overflow packet parallelism partition payload pipeline pointer port process protocol proxy queue recursion redundancy registry replication runtime scheduler segmentation serialization server shell snapshot socket stack storage subnet swap synchronization syntax syscall terminal thread throughput token topology virtualization vmware websocket workload abstraction annotation api argument array assertion asynchronous attribute backend boilerplate bootstrap callback camelcase class closure collection commit component constructor coroutine crud cursor dataclass deadlock decorator destructor devops dictionary encapsulation enumeration exception expression frontend function garbagecollection generic gitignore handler immutable inheritance initializer instance interface interpreter iterator javascript json keyword lambda linter loop method microservice mocking module mutex nan namespace nullability object observability overloading override package parameter parsing pathname polymorphism promise protobuf pseudocode python reactive refactoring repository responsive restful routing sandbox scalability script sdk singleton snippet software sql statement stylesheet syntaxerror template testing typescript undefined unittest validation variable versioning webhook workflow xml yaml accesscontrol authentication authorization backdoor botnet captcha certificate cipher crossorigin csrf cybersecurity ddos dns encryption exploit firewall forensics handshake honeypot https identity injection ipaddress malware mitigation phishing ransomware sandboxing spoofing spyware ssh ssl threatmodel tls vulnerability websecurity whitelist zeroday aggregation analytics apache backup bigquery cassandra clustering columnar consistency datamart dataset datastore denormalization etl hadoop hive ingestion joins kafka mariadb mongodb mysql normalization nosql optimization postgresql query redis schema sharding snowflake spark streaming transaction warehouse activation adapter agentic alignment artificialintelligence attention autoregressive backpropagation benchmark bias checkpoint classification clustering compression computervision contextwindow convergence corpus dataset decoder diffusion distillation embedding encoder entropy evaluation fewshot finetuning generation gradient grounding hallucination hyperparameter inference instructiontuning langchain latentspace llm lora machinelearning multimodal neuralnetwork normalization openweights optimization overfitting parameter perceptron pretraining prompt quantization rag ranking reasoning recognition recommendation regression reinforcementlearning retrieval rlhf sampling semanticsearch similarity supervised syntheticdata temperature tensor tokenization transformer unsupervised vectordatabase visionlanguage zeroshot ansible autoscaling availability azure buildpipeline cdn cicd cloudflare containerization dashboard digitalocean docker elasticity grafana helm infrastructure jenkins kubernetes loadtesting logstash monitoring observability openstack prometheus provisioning rollback scaling serverless terraform uptime vpc apt bash chmod chown cron curl debian environment fedora grep hostname iptables journalctl kubernetes linux mount nano nginx openssh permission processid sed ssh sudo systemd tmux ubuntu unix vim wget zsh'''
words = words_str.split()

In [42]:
words = list(set(words))

In [43]:
len(words)

409

In [38]:
for word in words:
    cursor.execute(
        'select id from words where word = ?',
        (word,)
    )
    result = cursor.fetchone()
    if not result:
        cursor_prod.execute(
            'select * from english_explain_dict where words=%s',
            (word,)
        )
        result = cursor_prod.fetchone()
        if result:
            phonetic_us = result['pronunciation']
            phonetic_uk = result['b_pronunciation']
            example_en = result['sentence']
            example_zh = result['sentence_translation']
            word_mean = json.loads(result['words_mean'])
            meanings = []
            for meaning in word_mean:
                pos = meaning['part'] if 'part' in meaning.keys() else ''
                meaning_cn = ';'.join(meaning['means'])
                meanings.append({
                    'pos': pos,
                    'meaning_cn': meaning_cn,
                })
            plural = result['word_plural']
            past_tense = result['past_tense']
            past_participle = result['past_participle']
            comparative = result['comparative']
            superlative = result['superlative']
            third_person = result['third_person_singular']

            # 使用 ? 作为占位符，并且不添加 indent
            # 将 INSERT ... SET 改为 INSERT INTO ... VALUES
            cursor.execute(
                'INSERT INTO words (phonetic_us, phonetic_uk, example_en, example_cn, meanings, plural, past_tense, past_participle, comparative, superlative, third_person, word) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)',
                (
                    phonetic_us,
                    phonetic_uk,
                    example_en,
                    example_zh,  # 注意：你的字段名是 example_cn，但变量是 example_zh
                    json.dumps(meanings, ensure_ascii=False, indent=1),
                    plural,
                    past_tense,
                    past_participle,
                    comparative,
                    superlative,
                    third_person,
                    word,
                )
            )

In [39]:
conn.commit()

In [46]:
book_id = 7
for word in words:
    cursor.execute(
        'select id from words where word = ?',
        (word,)
    )
    result = cursor.fetchone()
    if result:
        word_id = result[0]
        cursor.execute(
            'insert into word_book_words (word_book_id, word_id) values (?, ?)',
            (book_id, word_id,)
        )

In [47]:
conn.commit()

In [45]:
conn.rollback()

## 单词数据更新

In [7]:
sql = 'select * from words'
df_words = pd.read_sql(sql, conn)

In [10]:
df_words[:1]

,id,word,phonetic,pos,meaning_cn,category,frequency_rank,example_en,example_cn,word_book_id,...,audio_uk,audio_us,meanings,plural,past_tense,past_participle,present_participle,comparative,superlative,third_person
0,1,apple,,noun,苹果,food,3,She bit into a crisp red apple.,她咬了一口脆生生的红苹果。,1,...,/audio/uk/apple.mp3,/audio/us/apple.mp3,"[{""pos"": ""noun"", ""meaning_cn"": ""苹果""}]",apples,,,,,,


In [11]:
cursor_prod.execute(
    'select * from english_explain_dict where words=%s',
    ('name',)
)
result = cursor_prod.fetchone()
result

{'id': 10941,
 'deleted': 0,
 'creator_id': None,
 'creator': None,
 'create_time': datetime.datetime(2025, 6, 10, 14, 43, 7),
 'updater_id': None,
 'updater': None,
 'update_time': datetime.datetime(2026, 3, 20, 15, 52, 1),
 'uuid': None,
 'words': 'name',
 'words_type': None,
 'pronunciation': 'neɪm',
 'phonetic_transcription': None,
 'word_plural': '["names"]',
 'words_mean': '[{"means": ["名称", "命名", "确定", "任命", "给…取名", "叫出…的名字", "说出…的名称", "准确陈述"], "part": "vt."}, {"means": ["名称", "名字", "以…著名的", "名声", "有…名称的", "名誉", "名人", "名气", "有…名声的"], "part": "n."}, {"means": ["著名的", "(作品等)据以取名的"], "part": "adj."}]',
 'sentence': 'The name has come down from the last century.',
 'sentence_translation': '这名称是从上个世纪流传下来的。',
 'extended_meaning': None,
 'professional_definition': None,
 'third_person_singular': '[\n "names"\n]',
 'past_tense': '[\n "named"\n]',
 'past_participle': '[\n "named"\n]',
 'present_participle': '[\n "naming"\n]',
 'comparative': '[]',
 'superlative': '[]',
 'common_collocati

In [20]:
import json

for i, row in df_words.iterrows():
    word = row['word']
    cursor_prod.execute(
        'select * from english_explain_dict where words=%s',
        (word,)
    )
    result = cursor_prod.fetchone()
    if result:
        phonetic_us = result['pronunciation']
        phonetic_uk = result['b_pronunciation']
        example_en = result['sentence']
        example_zh = result['sentence_translation']
        word_mean = json.loads(result['words_mean'])
        meanings = []
        print(word_mean)
        for meaning in word_mean:
            pos = meaning['part'] if 'part' in meaning.keys() else ''
            meaning_cn = ';'.join(meaning['means'])
            meanings.append({
                'pos': pos,
                'meaning_cn': meaning_cn,
            })
        plural = result['word_plural']
        past_tense = result['past_tense']
        past_participle = result['past_participle']
        comparative = result['comparative']
        superlative = result['superlative']
        third_person = result['third_person_singular']

        # 使用 ? 作为占位符，并且不添加 indent
        cursor.execute(
            'update words set phonetic_us=?, phonetic_uk=?, example_en=?, example_cn=?, meanings=?, plural=?, past_tense=?, past_participle=?, comparative=?, superlative=?, third_person=? where word=?',
            (
                phonetic_us,
                phonetic_uk,
                example_en,
                example_zh,
                json.dumps(meanings, ensure_ascii=False, indent=1),  # 移除 indent=1
                plural,
                past_tense,
                past_participle,
                comparative,
                superlative,
                third_person,
                word,
            )
        )

[{'means': ['苹果'], 'part': 'n.'}]
[{'means': ['香蕉'], 'part': 'n.'}]
[{'means': ['橙色', '橘黄色的', '橙红色的', '奥兰治党的，奥兰治社团的(新教政治团体，认为北爱尔兰应继续为英国一部分)'], 'part': 'adj.'}, {'means': ['橙汁', '橙子', '柑橘', '橘黄色', '橙红色', '橘汁饮料'], 'part': 'n.'}]
[{'means': ['葡萄'], 'part': 'n.'}]
[{'part': 'n.', 'means': ['草莓', '草莓色']}]
[{'part': 'n.', 'means': ['西瓜']}]
[{'means': ['桃子', '桃', '桃红色', '粉红色', '极好的人(或物)', '特别漂亮的东西(或人)'], 'part': 'n.'}, {'means': ['粉红色的', '桃红色的'], 'part': 'adj.'}, {'means': ['<非正式>告密', '〈俚', '口〉揭发'], 'part': 'v.'}]
[{'means': ['柠檬', '柠檬汁', '柠檬色', '柠檬饮料', '浅黄色', '无用的东西', '蠢人'], 'part': 'n.'}, {'means': ['浅黄色的', '柠檬色的'], 'part': 'adj.'}]
[{'means': ['樱桃', '樱桃树', '樱花树', '樱桃木', '樱桃色'], 'part': 'n.'}, {'means': ['樱桃色的', '鲜红色的'], 'part': 'adj.'}]
[{'means': ['芒果'], 'part': 'n.'}]
[{'means': ['菠萝，凤梨'], 'part': 'n.'}]
[{'means': ['梨'], 'part': 'n.'}]
[{'means': ['蓝莓', '越橘蓝色浆果 (产于北美，可食)'], 'part': 'n.'}]
[{'means': ['椰子', '(用于烹调的)椰子肉，椰蓉'], 'part': 'n.'}]
[{'means': ['梅子', '李子', '紫红色'], 'part': 'n.'}, {

In [21]:
conn.commit()

## 添加总的单词库

In [31]:
sql = 'select * from english_explain_dict where deleted=0'
df_words = pd.read_sql(sql, conn_prod)

C:\Users\17245\AppData\Local\Temp\ipykernel_43352\756755393.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_words = pd.read_sql(sql, conn_prod)


In [32]:
len(df_words)

91064

In [33]:
df_words

,id,deleted,creator_id,creator,create_time,updater_id,updater,update_time,uuid,words,...,opposite,b_pronunciation,syllable,explain,source,book_source,check_status,checked_sentence,checked_sentence_translation,checked_words_mean
0,1379,0,None,None,2025-06-10 11:37:46,None,None,2026-03-20 14:56:33,None,zany,...,None,ˈzeɪni,None,"[{""pronunciation"": ""ˈzeɪni"", ""b_pronunciation""...",bd,None,NaN,The clown's zany jokes made all the children l...,小丑滑稽的笑话让所有的孩子都笑了。,"[\n {\n ""means"": [\n ""古怪的"",\n ""滑稽可笑的""\n ..."
1,1380,0,None,None,2025-06-10 11:37:46,None,None,2026-03-20 15:03:16,None,zanily,...,None,,None,"[{""pronunciation"": """", ""b_pronunciation"": """", ...",bd,None,NaN,The clown acted zanily to make the children la...,小丑表现得滑稽可笑，逗得孩子们哈哈大笑。,"[\n {\n ""means"": [\n ""滑稽的"",\n ""荒唐可笑地""\n ..."
2,1381,0,None,None,2025-06-10 11:37:46,None,None,2026-03-20 15:03:19,None,zaniness,...,None,,None,"[{""pronunciation"": """", ""b_pronunciation"": """", ...",bd,None,NaN,The clown's zaniness made all the children lau...,小丑的滑稽逗得所有的孩子开心地大笑。,"[\n {\n ""means"": [\n ""滑稽""\n ],\n ""part_na..."
3,1382,0,None,None,2025-06-10 11:37:46,None,None,2026-03-20 15:03:22,None,zap,...,None,zæp,None,"[{""pronunciation"": ""zæp"", ""b_pronunciation"": ""...",bd,None,NaN,I used the remote to zap the TV channel.,我用遥控器切换了电视频道。,"[\n {\n ""means"": [\n ""（使沿某方向）快速移动"",\n ""很快..."
4,1383,0,None,None,2025-06-10 11:37:46,None,None,2026-03-20 15:03:25,None,zappy,...,None,,None,"[{""pronunciation"": """", ""b_pronunciation"": """", ...",bd,None,NaN,The new video game is so zappy that I can't st...,这款新电子游戏非常刺激，让我停不下来。,"[\n {\n ""means"": [\n ""活泼的"",\n ""充满活力的"",\n ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91059,120135,0,None,None,2026-05-22 14:09:34,None,None,2026-05-22 17:57:23,None,far away from,...,"[\n ""close to"",\n ""near to""\n]",fɑː(r) əˈweɪ frəm,None,"[{""pronunciation"": ""fɑːr əˈweɪ frəm"", ""b_pronu...",None,None,10.0,My home is far away from my school.,我的家离我的学校很远。,"[\n {\n ""part"": null,\n ""means"": [\n ""远离；离..."
91060,120136,0,None,None,2026-05-22 14:09:40,None,None,2026-05-22 17:57:23,None,at the end of,...,[],ət ði end ɒv,None,"[{""pronunciation"": ""ət ði end əv"", ""b_pronunci...",None,None,10.0,"At the end of the film, the hero wept bitterly.",在影片的结尾，主人公伤心地哭了。,"[\n {\n ""part"": null,\n ""means"": [\n ""在…结尾..."
91061,120137,0,None,None,2026-05-22 14:09:53,None,None,2026-05-22 17:57:23,None,knock down,...,"[\n ""build up"",\n ""erect"",\n ""raise""\n]",ˈnɒk daʊn,None,"[{""pronunciation"": ""ˈnɑːk daʊn"", ""b_pronunciat...",None,None,10.0,The car knocked down a tree.,汽车撞倒了一棵树。,"[\n {\n ""part"": null,\n ""means"": [\n ""击倒；撞..."
91062,120138,0,None,None,2026-05-22 14:10:28,None,None,2026-05-22 17:57:23,None,pick ... up,...,[],pɪk ʌp,None,"[{""pronunciation"": ""pɪk ʌp"", ""b_pronunciation""...",None,None,10.0,She picked up the phone and dialed his number.,她拿起电话拨了他的号码。,"[\n {\n ""part"": null,\n ""means"": [\n ""拿起；捡..."


In [39]:
import json_repair
for _, row in df_words.iterrows():
    word = row['words']
    cursor.execute(
        'select id from words where word = ?',
        (word,)
    )
    # print(word)
    result = cursor.fetchone()
    if not result:
        phonetic_us = row['pronunciation']
        phonetic_uk = row['b_pronunciation']
        example_en = row['sentence']
        example_zh = row['sentence_translation']
        word_mean = json.loads(row['words_mean']) if not pd.isna(row['words_mean']) else None
        meanings = []
        try:
            for meaning in word_mean:
                pos = meaning['part'] if isinstance(meaning, dict) and 'part' in meaning.keys() else ''
                meaning_cn = ';'.join(meaning['means'])
                meanings.append({
                    'pos': pos,
                    'meaning_cn': meaning_cn,
                })
        except Exception as e:
            pass
        plural = row['word_plural']
        past_tense = row['past_tense']
        past_participle = row['past_participle']
        comparative = row['comparative']
        superlative = row['superlative']
        third_person = row['third_person_singular']

        # 使用 ? 作为占位符，并且不添加 indent
        # 将 INSERT ... SET 改为 INSERT INTO ... VALUES
        cursor.execute(
            'INSERT INTO words (phonetic_us, phonetic_uk, example_en, example_cn, meanings, plural, past_tense, past_participle, comparative, superlative, third_person, word) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)',
            (
                phonetic_us,
                phonetic_uk,
                example_en,
                example_zh,
                json.dumps(meanings, ensure_ascii=False, indent=1),
                plural,
                past_tense,
                past_participle,
                comparative,
                superlative,
                third_person,
                word,
            )
        )

In [38]:
conn.rollback()

In [40]:
conn.commit()

In [6]:
import pandas as pd
import json
from tqdm import tqdm  # 显示进度条

def fast_insert_words(df_words, cursor, conn, batch_size=500):
    """
    优化版本 - 适合10万条数据
    预计耗时：从几分钟降到几秒钟
    """

    # 1. 一次性获取所有已存在的词（避免逐条查询）
    words_list = df_words['words'].tolist()
    placeholders = ','.join(['?'] * len(words_list))

    cursor.execute(f'SELECT word FROM words WHERE word IN ({placeholders})', words_list)
    existing_words = {row[0] for row in cursor.fetchall()}

    # 2. 过滤出需要插入的数据（pandas 向量化过滤）
    mask = ~df_words['words'].isin(existing_words)
    df_new = df_words[mask].copy()

    print(f"总数据: {len(df_words)}, 已存在: {len(existing_words)}, 需插入: {len(df_new)}")

    if len(df_new) == 0:
        print("没有新数据需要插入")
        return 0

    # 3. 向量化处理 meanings（避免循环）
    def process_meanings_fast(meanings_json):
        try:
            word_mean = json.loads(meanings_json)
            meanings = []
            for meaning in word_mean:
                pos = meaning.get('part', '')
                meaning_cn = ';'.join(meaning['means'])
                meanings.append({'pos': pos, 'meaning_cn': meaning_cn})
            return json.dumps(meanings, ensure_ascii=False)
        except:
            return json.dumps([])

    # 使用 apply 比 iterrows 快很多
    df_new['meanings_processed'] = df_new['words_mean'].apply(process_meanings_fast)

    # 4. 准备批量插入的数据
    data_to_insert = []
    columns = ['pronunciation', 'b_pronunciation', 'sentence', 'sentence_translation',
               'meanings_processed', 'word_plural', 'past_tense', 'past_participle',
               'comparative', 'superlative', 'third_person_singular', 'words']

    # 使用 to_numpy() 转换为数组，比 iterrows 快100倍
    values = df_new[columns].to_numpy()

    # 5. 分批插入
    total_inserted = 0
    for i in tqdm(range(0, len(values), batch_size), desc="插入数据"):
        batch = values[i:i+batch_size].tolist()
        cursor.executemany('''
            INSERT INTO words (
                phonetic_us, phonetic_uk, example_en, example_cn,
                meanings, plural, past_tense, past_participle,
                comparative, superlative, third_person, word
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', batch)
        conn.commit()
        total_inserted += len(batch)

    return total_inserted

# 使用
inserted = fast_insert_words(df_words, cursor, conn)
print(f"成功插入 {inserted} 条数据")

总数据: 91064, 已存在: 0, 需插入: 91064


插入数据:   0%|          | 0/183 [00:05<?, ?it/s]


OperationalError: database is locked